# Game Discount Processing Notebook
This notebook implements Step 1 and Step 2 of the coding test for student discount calculations based on game play events.

In [1]:
# Step 1: Convert Input_Game_Events.json to Output_Game_Events_Discounts.csv
import json
import csv
from datetime import datetime

def parse_time(t):
    return datetime.strptime(t, "%Y-%m-%dT%H:%M:%S.%fZ")

def compute_play_minutes(data):
    total_seconds = 0
    no_of_responses = 0
    successful_responses = 0

    for round_data in data:
        start = parse_time(round_data['roundStartTime'])
        end = parse_time(round_data['roundEndTime'])
        total_seconds += (end - start).total_seconds()

        responses = round_data.get('responseText', [])
        target = round_data.get('targetText', [])
        no_of_responses += len(responses)
        successful_responses += sum(1 for r in responses if r in target)

    return total_seconds / 60, no_of_responses, successful_responses

def step1_generate_csv(json_file, output_csv):
    with open(json_file, 'r') as f:
        events_json = json.load(f)

    with open(output_csv, 'w', newline='') as out_csv:
        writer = csv.writer(out_csv)
        writer.writerow(['EventId', 'StudentId', 'EventType', 'PlayMinutes', 'NoOfResponses', 'NoOfSuccessfulResponses'])

        for event in events_json['events']:
            event_id = event['eventId']
            student_id = event['studentId']
            playset_type = event['playsetType']
            event_type = 'DH' if playset_type == 'H' else 'DS'

            play_minutes, responses, successful = compute_play_minutes(event['data'])

            writer.writerow([event_id, student_id, event_type, round(play_minutes, 2), responses, successful])

In [2]:
step1_generate_csv(r"C:/Users/RAVI/Downloads/Data Scientist Tests/Data Scientist Tests/Use Case 1/Input_Game_Events.json",
                   r"C:/Users/RAVI/Downloads/Data Scientist Tests/Data Scientist Tests/Use Case 1/output_Game_Events_Discounts.csv")

In [6]:
# Step 2: Update Discounts Based on Events
import pandas as pd

def step2_update_discounts(start_file, events_file, output_file):
    discounts_df = pd.read_csv()
    events_df = pd.read_csv()

    for _, row in events_df.iterrows():
        student_id = row['StudentId']
        event_type = row['EventType']
        play_minutes = row['PlayMinutes']

        matching_idxs = discounts_df[discounts_df['StudentId'] == student_id].index
        for idx in matching_idxs:
            playset_type = discounts_df.at[idx, 'PlaysetType']

            if event_type == 'DS':
                if playset_type == 'S':
                    discounts_df.at[idx, 'DiscountMinutes'] += play_minutes
                else:
                    discounts_df.at[idx, 'DiscountMinutes'] -= play_minutes
            elif event_type == 'DH':
                if playset_type == 'H':
                    discounts_df.at[idx, 'DiscountMinutes'] += play_minutes
                else:
                    discounts_df.at[idx, 'DiscountMinutes'] -= play_minutes

    # Save updated discounts to file
    discounts_df.to_csv("C:/Users/RAVI/Downloads/Data Scientist Tests/Data Scientist Tests/Use Case 1/output_discount_Events.csv", index=False)

    # Find min/max discounts
    student_grouped = discounts_df.groupby('StudentId')['DiscountMinutes'].sum()
    min_student = student_grouped.idxmin()
    max_student = student_grouped.idxmax()

    print(f"Minimum Discount: {min_student}, {student_grouped[min_student]}")
    print(f"Maximum Discount: {max_student}, {student_grouped[max_student]}")

In [7]:
import pandas as pd

def update_discounts(start_file, events_file, output_file):
    # Load data
    discounts_df = pd.read_csv("C:/Users/RAVI/Downloads/Data Scientist Tests/Data Scientist Tests/Use Case 1/Input_Start_Student_Discounts.txt")
    events_df = pd.read_csv("C:/Users/RAVI/Downloads/Data Scientist Tests/Data Scientist Tests/Use Case 1/output_Game_Events_Discounts.csv")

    # Merge data for efficient processing
    merged_df = discounts_df.merge(events_df, on='StudentId', how='left')

    # Update discounts based on event type and playset type
    conditions_ds = (merged_df['EventType'] == 'DS') & (merged_df['PlaysetType'] == 'S')
    conditions_dh = (merged_df['EventType'] == 'DH') & (merged_df['PlaysetType'] == 'H')

    merged_df['DiscountMinutes'] += merged_df['PlayMinutes'].where(conditions_ds, -merged_df['PlayMinutes'])
    merged_df['DiscountMinutes'] += merged_df['PlayMinutes'].where(conditions_dh, -merged_df['PlayMinutes'])

    # Group by StudentId to find min/max discounts
    student_grouped = merged_df.groupby('StudentId')['DiscountMinutes'].sum()
    min_student = student_grouped.idxmin()
    max_student = student_grouped.idxmax()

    print(f"Minimum Discount: {min_student}, {student_grouped[min_student]}")
    print(f"Maximum Discount: {max_student}, {student_grouped[max_student]}")

    # Save updated discounts to file
    merged_df.to_csv(output_file, index=False)

# Example usage:
update_discounts(
    "C:/Users/RAVI/Downloads/Data Scientist Tests/Data Scientist Tests/Use Case 1/Input_Start_Student_Discounts.txt",
    "C:/Users/RAVI/Downloads/Data Scientist Tests/Data Scientist Tests/Use Case 1/output_Game_Events_Discounts.csv",
    "C:/Users/RAVI/Downloads/Data Scientist Tests/Data Scientist Tests/Use Case 1/output_discount_Events.csv"
)

Minimum Discount: MS000100, -23.08
Maximum Discount: AS000200, 2.1400000000000006
